## df = spark.read.format('delta').load(f"schema.tabela")

Gera um dataframe para cada tabela delta de bronze.


In [0]:
df_games     = spark.read.format("delta").table("bronze.metacritic_games")
df_steam     = spark.read.format("delta").table("bronze.steam_metrics")
df_sales     = spark.read.format("delta").table("bronze.regional_sales")
df_hltob     = spark.read.format("delta").table("bronze.how_long_to_beat")
df_links     = spark.read.format("delta").table("bronze.walkthrough_links")

## df = df_games.withColumn("nome_coluna", "valor")

Adiciona uma nova coluna (metadado) de data e hora de processamento e nome do arquivo de origem.

In [0]:
from pyspark.sql.functions import current_timestamp, lit

df_games     = df_games.withColumn("datetime_silver", current_timestamp()).withColumn("table_name", lit("metacritic_games"))
df_steam     = df_steam.withColumn("datetime_silver", current_timestamp()).withColumn("table_name", lit("steam_metrics"))
df_sales     = df_sales.withColumn("datetime_silver", current_timestamp()).withColumn("table_name", lit("regional_sales"))
df_hltob     = df_hltob.withColumn("datetime_silver", current_timestamp()).withColumn("table_name", lit("how_long_to_beat"))
df_links     = df_links.withColumn("datetime_silver", current_timestamp()).withColumn("table_name", lit("walkthrough_links"))

## df_apolice.write.format('delta').saveAsTable("nome_tabela")

Salva os dataframes em arquivos delta lake (formato de arquivo) no schema/database "bronze". As tabelas geradas são do tipo MANAGED (gerenciadas).

In [0]:
# df_apolice.write.format('delta').mode("overwrite").saveAsTable("bronze.apolice")
# df_carro.write.format('delta').mode("overwrite").saveAsTable("bronze.carro")
# df_cliente.write.format('delta').mode("overwrite").saveAsTable("bronze.cliente")
# df_endereco.write.format('delta').mode("overwrite").saveAsTable("bronze.endereco")
# df_estado.write.format('delta').mode("overwrite").saveAsTable("bronze.estado")
# df_marca.write.format('delta').mode("overwrite").saveAsTable("bronze.marca")
# df_modelo.write.format('delta').mode("overwrite").saveAsTable("bronze.modelo")
# df_municipio.write.format('delta').mode("overwrite").saveAsTable("bronze.municipio")
# df_regiao.write.format('delta').mode("overwrite").saveAsTable("bronze.regiao")
# df_sinistro.write.format('delta').mode("overwrite").saveAsTable("bronze.sinistro")
# df_telefone.write.format('delta').mode("overwrite").saveAsTable("bronze.telefone")

## Maiusculas, tirando siglas, etc e gravando no formato delta no Silver

Aplicando Data Quality

In [0]:
import re

from pyspark.sql import functions as F

# ---------- Helpers ----------
snake_upper_case = lambda x: re.sub(r"(?<=[a-z0-9])(?=[A-Z])|(?<=[A-Z])(?=[A-Z][a-z])", "_", x).upper()

def _apply_name_rules(colname: str) -> str:
    """Regras de renome: upper + prefixos 'CD_', 'VL_', etc."""
    n = snake_upper_case(colname)
    n = n.replace("NA_", "NORTH_AMERICA_")
    n = n.replace("EU_", "EUROPE_")
    n = n.replace("JP_", "JAPAN_")
    n = n.replace("OTHER_", "OTHER_REGIONAL_")
    n = n.replace("JOGO", "GAME")
    return n

def _safe_drop(df, cols):
    """Dropa colunas somente se existirem."""
    existing = set(df.columns)
    to_drop = [c for c in cols if c in existing]
    return df.drop(*to_drop) if to_drop else df

# ---------- Core ----------
def rename_managed_colunms(src_fqn: str, dest_fqn: str = None):
    """
    Lê uma managed table (Delta) do metastore, aplica regras de renome,
    ajusta colunas de auditoria e salva como **managed table** via saveAsTable.
    - src_fqn: 'schema.tabela' de origem (ex.: 'bronze.games')
    - dest_fqn: 'schema.tabela' de destino; se None, sobrescreve a própria origem
    """
    dest_fqn = dest_fqn or src_fqn

    # Lê como TABELA (managed)
    df = spark.read.format("delta").table(src_fqn)

    # Renomeia todas as colunas de uma vez (evita conflito de rename em loop)
    new_cols = [_apply_name_rules(c) for c in df.columns]
    df = df.toDF(*new_cols)

    # Remove colunas antigas, se existirem
    df = _safe_drop(df, ["DATETIME_BRONZE", "FILENAME"])

    # Adiciona colunas de auditoria pedidas
    df = (df
          .withColumn("FILENAME_BRONZE", F.lit(src_fqn))     # origem rastreável
          .withColumn("FILEDATE_SILVER", F.current_timestamp())
         )

    # Salva como **Managed Table** (sem LOCATION) — sobrescrevendo destino
    (df.write
       .format("delta")
       .mode("overwrite")
       .saveAsTable(dest_fqn))

    return dest_fqn



In [0]:
rename_managed_colunms("bronze.metacritic_games",   "silver.metacritic_games")
rename_managed_colunms("bronze.steam_metrics",     "silver.steam_metrics")
rename_managed_colunms("bronze.regional_sales",   "silver.regional_sales")
rename_managed_colunms("bronze.how_long_to_beat",  "silver.how_long_to_beat")
rename_managed_colunms("bronze.walkthrough_links",    "silver.walkthrough_links")



'silver.walkthrough_links'

## (SQL) SHOW TABLES IN silver

Verifica os dados gravados no formato delta lake tipo MANAGED na camada silver.

In [0]:
%sql
SHOW TABLES IN silver

database,tableName,isTemporary
silver,how_long_to_beat,false
silver,metacritic_games,false
silver,regional_sales,false
silver,steam_metrics,false
silver,walkthrough_links,false


## (SQL) DESCRIBE DETAIL nome_tabela;


Vendo os detalhes de um tabela delta lake.

In [0]:
%sql
DESCRIBE DETAIL silver.metacritic_games;


format,id,name,description,location,createdAt,lastModified,partitionColumns,clusteringColumns,numFiles,sizeInBytes,properties,minReaderVersion,minWriterVersion,tableFeatures,statistics,clusterByAuto
delta,8f9f5458-a7a5-45eb-9a1a-5f1f38e9d717,workspace.silver.metacritic_games,null,,2026-10-20T19:27:57.357Z,2026-10-20T19:28:00.000Z,List(),List(),1,439786,"Map(delta.parquet.compression.codec -> zstd, delta.parquet.format.version.afe.internal -> 2.12.0, delta.enableDeletionVectors -> true, delta.parquet.format.version -> 2.12.0, delta.enableRowTracking -> true, delta.checkpointPolicy -> v2, io.unitycatalog.tableId -> 3cfc8220-8805-4928-bd8a-f40aa8b8f156, delta.rowTracking.materializedRowCommitVersionColumnName -> _row-commit-version-col-5db4b935-f768-4080-93b7-19628071563a, delta.rowTracking.materializedRowIdColumnName -> _row-id-col-1270832b-1bcd-4621-9e10-2151fc31008b)",3,7,"List(appendOnly, deletionVectors, domainMetadata, invariants, rowTracking, v2Checkpoint)","Map(numRowsDeletedByDeletionVectors -> 0, numDeletionVectors -> 0)",false


## (SQL) DESCRIBE EXTENDED nome_tabela;
ou 
##(SQL) DESCRIBE TABLE EXTENDED nome_tabela;

Mostra se a tabela é MANAGED Ou EXTERNAL.


In [0]:
%sql
DESCRIBE EXTENDED silver.metacritic_games;
--DESCRIBE TABLE EXTENDED metacritic_games_bronze;

col_name,data_type,comment
ID,string,null
NAME,string,null
PLATFORM,string,null
RELEASE_DATE,string,null
METASCORE,string,null
USER_SCORE,string,null
DEVELOPER,string,null
PUBLISHER,string,null
GENRE,string,null
FILENAME_BRONZE,string,null
